# 基于 Transformer Encoder (Multi-Head Self-Attention) 的电影评论情感分析 (全量数据版)

本 Notebook 演示如何根据 Transformer 架构思路，使用 PyTorch **从零手写搭建** 一个基于 **Transformer Encoder** 的文本情感分类系统。

### 🌟 本 Notebook 的核心架构亮点：
1. **多头自注意力机制 (Multi-Head Self-Attention)**：采用 **4 头 (heads=4)** 注意力，并为 Query, Key, Value 分别显式构造独立的投影矩阵 ($W_Q, W_K, W_V$)。
2. **位置编码 (Positional Encoding)**：结合正弦/余弦 (Sinusoidal) 位置编码，赋予模型对文本词序与位置的感知能力。
3. **Transformer Encoder Block**：包含多头注意力、Feed-Forward 残差连接 (Residual Connections) 与 Layer Normalization (Add & Norm)。
4. **Mask 掩码与全局池化 (Masked Global Max Pooling)**：精确掩码 `<PAD>` 填充符，通过 Pooling 抽取全局文本最强语义特征，最终送入全连接层 (FC) 映射到 0/1 二分类。
5. **全量数据训练与 Early Stopping**：在全量 IMDB 数据集上按 85:15 划分训练/验证集，实时监控 `val_loss` 保存最佳模型权重 `best_transformer_model.pt`。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Change the current working directory
os.chdir('/content/drive/MyDrive/Colab_Data/RNN评论情感分析')

# Verify the current working directory
print(f"Current working directory: {os.getcwd()}")

Current working directory: /content/drive/MyDrive/Colab_Data/RNN评论情感分析


## 1. 导入必要的 Python 库与全量均衡数据采样

In [ ]:
import os
import re
import collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# 设置随机种子保证可复现性
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

def load_balanced_data(file_path, sample_per_class=None, random_seed=42):
    df = pd.read_csv(file_path)
    df_pos = df[df['真实标签'] == 1]
    df_neg = df[df['真实标签'] == 0]
    if sample_per_class is not None:
        pos_sampled = df_pos.sample(n=min(sample_per_class, len(df_pos)), random_state=random_seed)
        neg_sampled = df_neg.sample(n=min(sample_per_class, len(df_neg)), random_state=random_seed)
    else:
        min_len = min(len(df_pos), len(df_neg))
        pos_sampled = df_pos.sample(n=min_len, random_state=random_seed)
        neg_sampled = df_neg.sample(n=min_len, random_state=random_seed)
    return pd.concat([pos_sampled, neg_sampled]).sample(frac=1, random_state=random_seed).reset_index(drop=True)

train_path = 'data/data_train.csv'
test_path = 'data/data_test.csv'

df_raw_train = load_balanced_data(train_path, sample_per_class=None)
df_test = load_balanced_data(test_path, sample_per_class=None)

# 按 85:15 划分 Train 集与 Validation 集
df_train, df_val = train_test_split(df_raw_train, test_size=0.15, random_state=42, stratify=df_raw_train['真实标签'])

print(f"训练集样本数量: {len(df_train)} (正面: {sum(df_train['真实标签']==1)}, 负面: {sum(df_train['真实标签']==0)})")
print(f"验证集样本数量: {len(df_val)}   (正面: {sum(df_val['真实标签']==1)}, 负面: {sum(df_val['真实标签']==0)})")
print(f"测试集样本数量: {len(df_test)}  (正面: {sum(df_test['真实标签']==1)}, 负面: {sum(df_test['真实标签']==0)})")
display(df_train.head())

训练集样本数量: 21250 (正面: 10625, 负面: 10625)
验证集样本数量: 3750   (正面: 1875, 负面: 1875)
测试集样本数量: 25000  (正面: 12500, 负面: 12500)


,影评内容,真实标签
11904,I only watched the first 30 minutes of this an...,0
19929,Its no surprise that Busey later developed a t...,1
15180,This is the second movie I saw for Horrorfest ...,0
12711,"i read the book ""7 years in Tibet"" from Heinri...",0
4295,I don't know what the Oscar voters saw in this...,0


## 2. 文本清洗、词表构建与预训练词向量加载

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return []
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    tokens = text.lower().split()
    return tokens

train_tokens = [clean_text(text) for text in df_train['影评内容']]
val_tokens = [clean_text(text) for text in df_val['影评内容']]
test_tokens = [clean_text(text) for text in df_test['影评内容']]

def build_vocab(tokenized_texts, max_vocab_size=15000):
    counter = collections.Counter()
    for tokens in tokenized_texts:
        counter.update(tokens)

    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, _ in counter.most_common(max_vocab_size - 2):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(train_tokens, max_vocab_size=15000)
print(f"词表构建完成，限制上限大小为: {len(vocab)}")

def load_pretrained_embeddings(vocab, embed_dim=128, glove_path=None):
    vocab_size = len(vocab)
    embedding_matrix = np.random.normal(scale=1.0 / np.sqrt(embed_dim), size=(vocab_size, embed_dim))
    embedding_matrix[vocab['<PAD>']] = np.zeros(embed_dim)

    if glove_path and os.path.exists(glove_path):
        print(f"正在加载 GloVe 词向量: {glove_path}...")
        loaded_count = 0
        with open(glove_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                word = parts[0]
                if word in vocab:
                    vector = np.array(parts[1:], dtype=np.float32)
                    if len(vector) == embed_dim:
                        embedding_matrix[vocab[word]] = vector
                        loaded_count += 1
        print(f"成功加载 {loaded_count}/{vocab_size} 个预训练词向量。")
    else:
        print("未找到本地 GloVe 词向量，采用符合标准缩放高斯分布的方差矩阵初始化。")
    return torch.tensor(embedding_matrix, dtype=torch.float32)

pretrained_weight = load_pretrained_embeddings(vocab, embed_dim=128, glove_path='data/glove.6B.100d.txt')

词表构建完成，限制上限大小为: 15000
未找到本地 GloVe 词向量，采用符合标准缩放高斯分布的方差矩阵初始化。


## 3. 序列 Padding 与 PyTorch Dataset / DataLoader 构建

In [ ]:
def encode_and_pad(tokens, vocab, max_len=200):
    seq = [vocab.get(word, vocab['<UNK>']) for word in tokens]
    if len(seq) < max_len:
        seq = seq + [vocab['<PAD>']] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    return seq

class MovieReviewDataset(Dataset):
    def __init__(self, tokenized_texts, labels, vocab, max_len=200):
        self.samples = [encode_and_pad(tokens, vocab, max_len) for tokens in tokenized_texts]
        self.labels = labels

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return torch.tensor(self.samples[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float32)

max_len = 200
batch_size = 64

train_dataset = MovieReviewDataset(train_tokens, df_train['真实标签'].values, vocab, max_len)
val_dataset = MovieReviewDataset(val_tokens, df_val['真实标签'].values, vocab, max_len)
test_dataset = MovieReviewDataset(test_tokens, df_test['真实标签'].values, vocab, max_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

Train batches: 333 | Val batches: 59 | Test batches: 391


## 4. 手写搭建 Transformer Encoder 模型 (4 头注意力 + QKV 独立投影矩阵)

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """
    4 头自注意力机制 (Multi-Head Self-Attention)
    显式采用 3 个独立的线性投影矩阵 (W_q, W_k, W_v) 分别变换 Query, Key, Value
    """
    def __init__(self, embed_dim=128, num_heads=8, dropout=0.1):
        super(MultiHeadSelfAttention, self).__init__()
        assert embed_dim % num_heads == 0, "embed_dim 必须能够被 num_heads 整除"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # 独立的 Q, K, V 投影矩阵
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)

        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: [batch_size, seq_len, embed_dim]
        batch_size, seq_len, embed_dim = x.size()

        # 通过各自独立的投影矩阵映射并划分为 4 个头
        # 映射后形状: [batch_size, num_heads, seq_len, head_dim]
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # 计算 Scaled Dot-Product Attention 得分: Q * K^T / sqrt(d_k)
        # scores 形状: [batch_size, num_heads, seq_len, seq_len]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            # mask 形状: [batch_size, 1, 1, seq_len]
            scores = scores.masked_fill(mask == 0, -1e9)

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 加权求和得到上下文向量: [batch_size, num_heads, seq_len, head_dim]
        context = torch.matmul(attn_weights, V)

        # 拼接 4 个头的特征: [batch_size, seq_len, embed_dim]
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_dim)

        return self.out_proj(context)

class PositionalEncoding(nn.Module):
    """ 正弦/余弦位置编码 (Sinusoidal Positional Encoding) """
    def __init__(self, embed_dim=128, max_len=500):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-np.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # [1, max_len, embed_dim]
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerEncoderBlock(nn.Module):
    """ 单层 Transformer Encoder 模块 """
    def __init__(self, embed_dim=128, num_heads=4, dim_feedforward=256, dropout=0.1):
        super(TransformerEncoderBlock, self).__init__()
        self.self_attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, embed_dim)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Sub-layer 1: Self-Attention + Residual + LayerNorm
        attn_out = self.self_attn(x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        # Sub-layer 2: Feed Forward + Residual + LayerNorm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x

class SentimentTransformer(nn.Module):
    """ 基于 Transformer Encoder 的电影评论情感分析二分类模型 """
    def __init__(self, vocab_size, embed_dim=128, num_heads=4, num_layers=2, dim_feedforward=256, dropout=0.3, pad_idx=0, pretrained_weight=None):
        super(SentimentTransformer, self).__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        if pretrained_weight is not None:
            self.embedding.weight.data.copy_(pretrained_weight)

        self.pos_encoder = PositionalEncoding(embed_dim=embed_dim)

        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, x):
        # x: [batch_size, seq_len]
        # 构造 Padding Mask (有效 token 为 1，PAD 为 0)
        mask = (x != self.pad_idx).unsqueeze(1).unsqueeze(2) # [batch_size, 1, 1, seq_len]

        out = self.embedding(x)
        out = self.pos_encoder(out)

        for layer in self.encoder_layers:
            out = layer(out, mask)

        # Masked Global Max Pooling 掩码全局最大池化
        # 避免 <PAD> 位置影响最大池化结果
        mask_expanded = (x != self.pad_idx).unsqueeze(-1).expand_as(out)
        out_masked = out.masked_fill(~mask_expanded, -1e9)
        out_pooled = torch.max(out_masked, dim=1)[0]

        out_pooled = self.dropout(out_pooled)
        logits = self.fc(out_pooled).squeeze(1)
        return logits

# 别名设置
TransformerClassifier = SentimentTransformer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SentimentTransformer(
    vocab_size=len(vocab),
    embed_dim=128,
    num_heads=4,        # 4头自注意力
    num_layers=2,       # 2层 Encoder Block
    dim_feedforward=256,
    dropout=0.3,
    pretrained_weight=pretrained_weight
).to(device)

print(model)

SentimentTransformer(
  (embedding): Embedding(15000, 128, padding_idx=0)
  (pos_encoder): PositionalEncoding()
  (encoder_layers): ModuleList(
    (0-1): 2 x TransformerEncoderBlock(
      (self_attn): MultiHeadSelfAttention(
        (W_q): Linear(in_features=128, out_features=128, bias=True)
        (W_k): Linear(in_features=128, out_features=128, bias=True)
        (W_v): Linear(in_features=128, out_features=128, bias=True)
        (out_proj): Linear(in_features=128, out_features=128, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
      )
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=128, out_features=256, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.3, inplace=False)
        (3): Linear(in_features=256, out_features=128, bias=True)
      )
      (dropout): Dropout(p=0.3, inplace=False)
    )
  )
  (dropout): Dro

## 5. Early Stopping 监控与训练循环

In [ ]:
class EarlyStopping:
    def __init__(self, patience=4, verbose=True, save_path='data/best_transformer_model.pt'):
        self.patience = patience
        self.verbose = verbose
        self.save_path = save_path
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping 计数: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, model):
        os.makedirs(os.path.dirname(self.save_path), exist_ok=True)
        torch.save(model.state_dict(), self.save_path)
        if self.verbose:
            print(f"验证集 Loss 改善，已保存最佳权重至 {self.save_path}")

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)
early_stopping = EarlyStopping(patience=4, verbose=True, save_path='data/best_transformer_model.pt')

epochs = 20
for epoch in range(1, epochs + 1):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        train_correct += (preds == targets).sum().item()
        train_total += targets.size(0)

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            logits = model(inputs)
            loss = criterion(logits, targets)
            val_loss += loss.item() * inputs.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            val_correct += (preds == targets).sum().item()
            val_total += targets.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total

    print(f"Epoch {epoch:02d}/{epochs} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}%")

    early_stopping(epoch_val_loss, model)
    if early_stopping.early_stop:
        print("触发 Early Stopping 条件，终止训练！")
        break

Epoch 01/20 | Train Loss: 0.7026 | Train Acc: 49.84% | Val Loss: 0.6934 | Val Acc: 51.09%
验证集 Loss 改善，已保存最佳权重至 data/best_transformer_model.pt
Epoch 02/20 | Train Loss: 0.6947 | Train Acc: 50.73% | Val Loss: 0.6926 | Val Acc: 51.20%
验证集 Loss 改善，已保存最佳权重至 data/best_transformer_model.pt
Epoch 03/20 | Train Loss: 0.6302 | Train Acc: 59.82% | Val Loss: 0.3945 | Val Acc: 82.91%
验证集 Loss 改善，已保存最佳权重至 data/best_transformer_model.pt
Epoch 04/20 | Train Loss: 0.3307 | Train Acc: 86.31% | Val Loss: 0.3470 | Val Acc: 85.12%
验证集 Loss 改善，已保存最佳权重至 data/best_transformer_model.pt
Epoch 05/20 | Train Loss: 0.2150 | Train Acc: 92.07% | Val Loss: 0.3757 | Val Acc: 86.43%
EarlyStopping 计数: 1/4
Epoch 06/20 | Train Loss: 0.1498 | Train Acc: 94.72% | Val Loss: 0.3852 | Val Acc: 86.91%
EarlyStopping 计数: 2/4
Epoch 07/20 | Train Loss: 0.1023 | Train Acc: 96.69% | Val Loss: 0.4616 | Val Acc: 85.31%
EarlyStopping 计数: 3/4
Epoch 08/20 | Train Loss: 0.0707 | Train Acc: 97.88% | Val Loss: 0.5800 | Val Acc: 84.83%
EarlyS

## 6. 加载最佳模型并在测试集上进行终极评估

In [ ]:
model.load_state_dict(torch.load('data/best_transformer_model.pt'))
model.eval()

test_preds = []
test_probs = []
true_labels = df_test['真实标签'].values

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)
        logits = model(inputs)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        test_probs.extend(probs)
        test_preds.extend(preds)

acc = accuracy_score(true_labels, test_preds)
prec, rec, f1, _ = precision_recall_fscore_support(true_labels, test_preds, average='binary')
cm = confusion_matrix(true_labels, test_preds)

print("=== 测试集终极评估结果 (Transformer Encoder 4-Heads) ===")
print(f"准确率 (Accuracy) : {acc*100:.2f}%")
print(f"精确率 (Precision): {prec*100:.2f}%")
print(f"召回率 (Recall)   : {rec*100:.2f}%")
print(f"F1 得分 (F1 Score): {f1*100:.2f}%")
print("\n混淆矩阵 (Confusion Matrix):")
print(cm)

=== 测试集终极评估结果 (Transformer Encoder 4-Heads) ===
准确率 (Accuracy) : 82.87%
精确率 (Precision): 92.16%
召回率 (Recall)   : 71.86%
F1 得分 (F1 Score): 80.75%

混淆矩阵 (Confusion Matrix):
[[11736   764]
 [ 3518  8982]]


## 7. 导出预测结果至 CSV 文件

In [ ]:
output_df = pd.DataFrame({
    '影评内容': df_test['影评内容'],
    '真实标签': true_labels,
    '预测标签': test_preds,
    '预测概率': np.round(test_probs, 4)
})

output_csv_path = 'data/transformer_predictions.csv'
output_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
print(f"已成功将 Transformer 测试预测结果保存至: {output_csv_path}")
display(output_df.head(10))

已成功将 Transformer 测试预测结果保存至: data/transformer_predictions.csv


,影评内容,真实标签,预测标签,预测概率
0,I don't even like watching those late night ta...,1,0,0.4912
1,Jude law gives Keanu Reeves a run for his mone...,0,0,0.0109
2,I am a big fan of this film and found the TV m...,1,1,0.5999
3,It's dreadful rubbish. I liked 'How Do You Wan...,0,0,0.0182
4,"Ron Howard and his ""editors"" only had one job ...",0,0,0.1830
5,I just got back from the GLBT Film Festival at...,1,1,0.7510
6,John Huston made many remarkable and memorable...,1,1,0.6851
7,I want the 99 minutes of my life back that was...,0,0,0.0078
8,You looking for a comic drama with suspense an...,1,1,0.9749
9,I caught this on local Mexican television at 2...,1,1,0.8356
